# pod-clone 환경을 Colab에 맞추기

`pod-clone:cpu` Docker 이미지(= SageMaker Pod 스택)의 패키지를 이 Colab 런타임에 설치해
환경을 동일하게 맞춥니다. **Runtime → Run all** 만 누르면 됩니다.

- 기준: Python 3.11 / torch 2.4.0 / transformers 4.49.0 / langchain 1.2.10 / langgraph 1.0.10 ...
- GPU가 필요하면 먼저 **Runtime → Change runtime type → T4 GPU** 로 바꾼 뒤 실행하세요.

> ⚠️ 설치 후 numpy/torch 등이 다운그레이드되면 **Runtime → Restart session** 후
> 검증 셀부터 다시 실행해야 버전이 제대로 반영됩니다.

## 0. 설치 전 현재 Colab 환경 확인

In [ ]:
import sys
print('python', sys.version.split()[0])
try:
    import torch
    print('torch (설치 전)', torch.__version__, '| cuda?', torch.cuda.is_available())
except Exception as e:
    print('torch 미설치:', e)
!nvidia-smi -L 2>/dev/null || echo 'GPU 없음 (CPU 런타임)'

## 1. requirements 파일 작성 (이미지에서 추출한 131개 패키지)

In [ ]:
%%writefile requirements-image.txt
accelerate==1.2.0
aiohappyeyeballs==2.6.2
aiohttp==3.14.1
aiosignal==1.4.0
annotated-doc==0.0.4
annotated-types==0.7.0
anthropic==0.84.0
anyio==4.14.0
attrs==26.1.0
backoff==2.2.1
bcrypt==5.0.0
build==1.5.0
certifi==2026.6.17
cffi==2.0.0
charset-normalizer==3.4.7
chromadb==1.3.6
click==8.4.1
cryptography==49.0.0
datasets==3.2.0
dill==0.3.8
distro==1.9.0
docstring_parser==0.18.0
durationpy==0.10
filelock==3.29.0
filetype==1.2.0
flatbuffers==25.12.19
frozenlist==1.8.0
fsspec==2024.9.0
google-auth==2.55.0
google-genai==1.75.0
googleapis-common-protos==1.75.0
grpcio==1.81.1
h11==0.16.0
httpcore==1.0.9
httptools==0.8.0
httpx==0.28.1
huggingface-hub==0.26.5
idna==3.18
importlib_resources==7.1.0
Jinja2==3.1.6
jiter==0.15.0
jsonpatch==1.33
jsonpointer==3.1.1
jsonschema==4.26.0
jsonschema-specifications==2025.9.1
kubernetes==36.0.2
langchain==1.2.10
langchain-anthropic==1.3.4
langchain-chroma==1.0.0
langchain-core==1.2.17
langchain-google-genai==4.2.1
langchain-openai==1.1.10
langgraph==1.0.10
langgraph-checkpoint==4.0.1
langgraph-prebuilt==1.0.8
langgraph-sdk==0.3.9
langsmith==0.7.13
markdown-it-py==4.2.0
MarkupSafe==3.0.3
mdurl==0.1.2
mmh3==5.2.1
mpmath==1.3.0
multidict==6.7.1
multiprocess==0.70.16
networkx==3.6.1
numpy==1.26.4
oauthlib==3.3.1
onnxruntime==1.27.0
openai==2.26.0
opentelemetry-api==1.42.1
opentelemetry-exporter-otlp-proto-common==1.42.1
opentelemetry-exporter-otlp-proto-grpc==1.42.1
opentelemetry-proto==1.42.1
opentelemetry-sdk==1.42.1
opentelemetry-semantic-conventions==0.63b1
orjson==3.11.9
ormsgpack==1.12.2
overrides==7.7.0
packaging==26.2
pandas==3.0.3
pillow==12.2.0
posthog==5.4.0
propcache==0.5.2
protobuf==6.33.6
psutil==7.2.2
pyarrow==24.0.0
pyasn1==0.6.3
pyasn1_modules==0.4.2
pybase64==1.4.3
pycparser==3.0
pydantic==2.12.5
pydantic_core==2.41.5
Pygments==2.20.0
PyPika==0.51.1
pyproject_hooks==1.2.0
python-dateutil==2.9.0.post0
python-dotenv==1.2.2
PyYAML==6.0.3
referencing==0.37.0
regex==2026.5.9
requests==2.34.2
requests-oauthlib==2.0.0
requests-toolbelt==1.0.0
rich==15.0.0
rpds-py==2026.5.1
safetensors==0.8.0
shellingham==1.5.4
six==1.17.0
sniffio==1.3.1
sympy==1.14.0
tenacity==9.1.4
tiktoken==0.13.0
tokenizers==0.21.1
torch==2.4.0
torchaudio==2.4.0
torchvision==0.19.0
tqdm==4.68.3
transformers==4.49.0
typer==0.26.7
typing-inspection==0.4.2
typing_extensions==4.15.0
urllib3==2.7.0
uuid_utils==0.16.2
uvicorn==0.49.0
uvloop==0.22.1
watchfiles==1.2.0
websocket-client==1.9.0
websockets==16.0
xxhash==3.7.0
yarl==1.24.2
zstandard==0.25.0

## 2. 설치

Colab 기본 패키지를 이미지 버전으로 맞춥니다(일부 다운그레이드 포함). 수 분 걸릴 수 있습니다.

> torch는 GPU 런타임이면 PyPI 기본 CUDA 빌드가 설치됩니다. 정확히 cu124가 필요하면
> 아래 셀의 주석(인덱스 지정)을 참고하세요.

In [ ]:
%pip install -q -r requirements-image.txt

# (선택) torch를 정확히 +cu124 빌드로 맞추려면 위 대신/추가로:
# %pip install -q torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 \
#   --index-url https://download.pytorch.org/whl/cu124
print('설치 완료. numpy/torch가 바뀌었다면 Runtime > Restart session 후 아래 검증 셀부터 실행하세요.')

## 3. 검증 — 이미지 기준 버전과 일치하는지 확인

(다운그레이드가 있었다면 먼저 **Runtime → Restart session** 후 이 셀부터 실행)

In [ ]:
import sys, importlib.metadata as md

# pod-clone 이미지 기준 핵심 버전
EXPECTED = {
    'torch': '2.4.0', 'torchvision': '0.19.0', 'torchaudio': '2.4.0',
    'transformers': '4.49.0', 'tokenizers': '0.21.1', 'huggingface-hub': '0.26.5',
    'datasets': '3.2.0', 'accelerate': '1.2.0', 'numpy': '1.26.4', 'pydantic': '2.12.5',
    'langchain': '1.2.10', 'langchain-core': '1.2.17', 'langchain-openai': '1.1.10',
    'langchain-anthropic': '1.3.4', 'langgraph': '1.0.10', 'langgraph-checkpoint': '4.0.1',
    'langsmith': '0.7.13', 'openai': '2.26.0', 'anthropic': '0.84.0', 'chromadb': '1.3.6',
}

print('python', sys.version.split()[0], '(기준 이미지: 3.11.x)\n')
ok = bad = 0
for pkg, want in EXPECTED.items():
    try:
        got = md.version(pkg)
    except md.PackageNotFoundError:
        got = None
    mark = '✅' if got == want else '❌'
    if got == want: ok += 1
    else: bad += 1
    print(f'{mark} {pkg:22s} 기대 {want:12s} 실제 {got}')
print(f'\n일치 {ok} / 불일치 {bad}')

## 4. 스모크 테스트 — 실제 import & GPU

In [ ]:
import torch, transformers, datasets, accelerate
import langchain, langgraph, langsmith, openai, anthropic, chromadb
print('torch', torch.__version__, '| cuda?', torch.cuda.is_available(),
      '| device:', (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'))
print('transformers', transformers.__version__)
print('langchain', langchain.__version__)
print('OK: 핵심 라이브러리 import 성공')